# Feather Example Notebook

This notebook demonstrates how to communicate with the ESP32-C6 Feather running the S.P.I.R.I.T. Board firmware.

The Feather serves as a development testbed for network features without requiring the full Furby hardware. It supports:
- Ping (connectivity test)
- LED blink command
- RGB LED color control

## Quick Start

1. Flash the firmware to your Feather: `uv run pio run -e feather_usb -t upload`
2. Update `FEATHER_HOST` below with your Feather's IP address
3. Run the cells to test connectivity and LED control

## Setup

We'll use Python's socket library to communicate directly with the Feather's TCP server on port 5000.

In [ ]:
import socket
import time

# Configure your Feather's IP address here
FEATHER_HOST = "192.168.66.243"  # UPDATE THIS with your Feather's IP
FEATHER_PORT = 5000

def send_command(hex_string: str, timeout: float = 2.0) -> str:
    """
    Send a hex command to the Feather and return the response.
    
    Args:
        hex_string: Hex command string (e.g., "FF" for ping, "09FF0000" for red LED)
        timeout: Socket timeout in seconds
    
    Returns:
        Response string from the firmware
    """
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        sock.connect((FEATHER_HOST, FEATHER_PORT))
        sock.sendall((hex_string + "\n").encode())
        response = sock.recv(1024).decode().strip()
    return response

print(f"Feather configured at {FEATHER_HOST}:{FEATHER_PORT}")

## Test Connectivity

Send a ping command (`FF`) to verify the Feather is responding.

In [ ]:
# Ping the firmware (command 0xFF)
response = send_command("FF")
print(f"Ping response: {response}")

if response == "31337":
    print("Connection successful!")
else:
    print(f"Warning: Unexpected response (expected '31337')")

## Blink the LED

Command `08 N` blinks the NeoPixel white N times (where N is a single hex digit, 1-F).

In [ ]:
# Blink the LED 3 times (command 0x08 with count 3)
response = send_command("08 3")
print(f"Blink response: {response}")

In [ ]:
# Blink 5 times
response = send_command("08 5")
print(f"Blink response: {response}")

## Control the RGB LED

Command `09 RR GG BB` sets the NeoPixel to a specific RGB color.

- `RR` = Red value (00-FF)
- `GG` = Green value (00-FF)  
- `BB` = Blue value (00-FF)

Examples:
- `09 FF 00 00` = Red
- `09 00 FF 00` = Green
- `09 00 00 FF` = Blue
- `09 FF FF FF` = White
- `09 00 00 00` = Off

In [ ]:
# Set LED to red
response = send_command("09 FF 00 00")
print(f"Set red: {response}")

In [ ]:
# Set LED to green
response = send_command("09 00 FF 00")
print(f"Set green: {response}")

In [ ]:
# Set LED to blue
response = send_command("09 00 00 FF")
print(f"Set blue: {response}")

In [ ]:
# Turn LED off
response = send_command("09 00 00 00")
print(f"LED off: {response}")

## Helper Functions

Convenience functions for common operations.

In [ ]:
def set_rgb(r: int, g: int, b: int) -> str:
    """
    Set the NeoPixel to the specified RGB color.
    
    Args:
        r: Red value (0-255)
        g: Green value (0-255)
        b: Blue value (0-255)
    
    Returns:
        Response from firmware
    """
    r = max(0, min(255, r))
    g = max(0, min(255, g))
    b = max(0, min(255, b))
    return send_command(f"09 {r:02X} {g:02X} {b:02X}")

def led_off() -> str:
    """Turn off the LED."""
    return set_rgb(0, 0, 0)

def blink(count: int = 3) -> str:
    """
    Blink the LED white.
    
    Args:
        count: Number of blinks (1-15)
    """
    count = max(1, min(15, count))
    return send_command(f"08 {count:X}")

print("Helper functions defined: set_rgb(r,g,b), led_off(), blink(count)")

## Demo: Color Cycle

Cycle through several colors to demonstrate RGB control.

In [ ]:
# Color cycle demo
colors = [
    (255, 0, 0, "Red"),
    (255, 128, 0, "Orange"),
    (255, 255, 0, "Yellow"),
    (0, 255, 0, "Green"),
    (0, 255, 255, "Cyan"),
    (0, 0, 255, "Blue"),
    (128, 0, 255, "Purple"),
    (255, 0, 128, "Magenta"),
]

print("Starting color cycle...")
for r, g, b, name in colors:
    print(f"  {name}")
    set_rgb(r, g, b)
    time.sleep(0.5)

# Turn off when done
led_off()
print("Color cycle complete!")

## Command Reference

| Command | Format | Description |
|---------|--------|-------------|
| Ping | `FF` | Returns `31337` if connected |
| Blink | `08 N` | Blink LED N times (N = 1-F hex) |
| Set RGB | `09 RR GG BB` | Set LED color (values 00-FF) |

All commands are sent as ASCII hex strings over TCP port 5000.